### Data Understanding

Below are the steps taken to understand the data used in this exercise. 

In [102]:
#Importing libraries and loading dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/vehicles.csv')
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [103]:
df.describe(include='all')

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
count,4.268800e+05,426880,4.268800e+05,425675.000000,409234,421603,252776,249202,423867,4.224800e+05,418638,424324,265838,296313,120519,334022,296677,426880
unique,NaN,404,NaN,NaN,42,29649,6,8,5,NaN,6,3,118246,3,4,13,12,51
top,NaN,columbus,NaN,NaN,ford,f-150,good,6 cylinders,gas,NaN,clean,automatic,1FMJU1JT1HEA52352,4wd,full-size,sedan,white,ca
freq,NaN,3608,NaN,NaN,70985,8009,121456,94169,356209,NaN,405117,336524,261,131904,63465,87056,79285,50614
mean,7.311487e+09,NaN,7.519903e+04,2011.235191,NaN,NaN,NaN,NaN,NaN,9.804333e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,4.473170e+06,NaN,1.218228e+07,9.452120,NaN,NaN,NaN,NaN,NaN,2.138815e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,7.207408e+09,NaN,0.000000e+00,1900.000000,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,7.308143e+09,NaN,5.900000e+03,2008.000000,NaN,NaN,NaN,NaN,NaN,3.770400e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,7.312621e+09,NaN,1.395000e+04,2013.000000,NaN,NaN,NaN,NaN,NaN,8.554800e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,7.315254e+09,NaN,2.648575e+04,2017.000000,NaN,NaN,NaN,NaN,NaN,1.335425e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
df.columns
#A quick look at column name can eliminate at least one column. '
#VIN' operates as a secondary identifier, so we can exclude it. 
df = df.drop(columns=['VIN'])

df.columns

Index(['id', 'region', 'price', 'year', 'manufacturer', 'model', 'condition',
       'cylinders', 'fuel', 'odometer', 'title_status', 'transmission',
       'drive', 'size', 'type', 'paint_color', 'state'],
      dtype='object')

In [105]:
df.dtypes
#Only the dependent variable (price) and two of the independent variables (year, and odometer) are numerical features.

id                int64
region           object
price             int64
year            float64
manufacturer     object
model            object
condition        object
cylinders        object
fuel             object
odometer        float64
title_status     object
transmission     object
drive            object
size             object
type             object
paint_color      object
state            object
dtype: object

In [106]:
#Yeilds percent of missing data for each feature
missing_count=df.isnull().sum()/426880*100

missing_count.sort_values()

id               0.000000
region           0.000000
price            0.000000
state            0.000000
year             0.282281
transmission     0.598763
fuel             0.705819
odometer         1.030735
model            1.236179
title_status     1.930753
manufacturer     4.133714
type            21.752717
paint_color     30.501078
drive           30.586347
condition       40.785232
cylinders       41.622470
size            71.767476
dtype: float64

### Data Preparation

For features with less than 5% missing values, we will drop all observations with missing data. Any categorical feature with greater than 30% missing observations will be removed from the model. This includes 'paint_color', 'drive', 'condition', 'cylinders', and 'size'. Removing 'model' for having too many categories. Re-categorizing 'manufacturer' to 'ford', 'chevrolet', 'toyota', and 'other' (16%,12%,8%,<4%). However, I hypothesize that 'paint_color' is a major contributor to the purchase of a car, so we will keep and recategorize as 'white', 'black, 'silver', 'other', and 'missing'. We will also recategorize 'type' to include 'sedan', 'SUV', combine 'pickup' and 'truck', 'missing', and 'other'.

In [107]:
#Log Transform Price
df=df[df['price']>100]
df=df[df['price']<100000]
df['price']=np.log(df['price'])

In [108]:
#drop features with high amount of missing values
df = df.drop(columns=['drive','condition','cylinders','size','model','state'])
df.columns

Index(['id', 'region', 'price', 'year', 'manufacturer', 'fuel', 'odometer',
       'title_status', 'transmission', 'type', 'paint_color'],
      dtype='object')

In [109]:
#dropping missing observations for features with <2% of missing data
df = df.dropna(subset=['year','transmission','fuel','odometer','title_status'])

In [110]:
#Log-Transforming Odometer
df['log_odometer']=np.log(df['odometer']+1)
df=df.drop(columns=['odometer'])

In [111]:
#Replacing Year with Vehicle Age
df['vehicle_age']=2024-df['year']
df=df.drop(columns=['year'])

In [112]:
df['age_mileage'] = df['vehicle_age'] * df['log_odometer']

In [113]:
#preparing 'paint_color' feature
#add 'missing' lable to color
df['paint_color']=df['paint_color'].fillna('missing')
#recategorize to top three colors
def recategorize_color(x):
    if x=='white':
        return 'white'
    elif x=='black':
        return 'black'
    elif x=='silver':
        return 'silver'
    elif x=='missing':
        return 'missing'
    else:
        return 'other'
df['paint_color']=df['paint_color'].apply(recategorize_color)
df['paint_color'].value_counts()


paint_color
missing    107402
other      102343
white       70541
black       56930
silver      38961
Name: count, dtype: int64

In [114]:
#preparing 'type' feature
#add 'missing' lable to type
df['type']=df['type'].fillna('missing')
#combining to 'truck' and 'pickup', and recategorizing top 3 types
def recategorize_type(x):
    if x in ['truck','pickup']:
        return 'truck'
    elif x=='SUV':
        return 'SUV'
    elif x=='sedan':
        return 'sedan'
    elif x=='missing':
        return 'missing'
    else:
        return 'other'

df['type']=df['type'].apply(recategorize_type)
df['type'].value_counts()

type
missing    82825
other      81204
sedan      76489
truck      69216
SUV        66443
Name: count, dtype: int64

In [115]:
#preparing 'manufacturer' feature
#add 'missing' lable to type
df['manufacturer']=df['manufacturer'].fillna('missing')
#combining to 'truck' and 'pickup', and recategorizing top 3 types
def recategorize_mfg(x):
    if x=='ford':
        return 'ford'
    elif x=='chevrolet':
        return 'chevrolet'
    elif x=='toyota':
        return 'toyota'
    elif x=='missing':
        return 'missing'
    else:
        return 'other'

df['manufacturer']=df['manufacturer'].apply(recategorize_mfg)
df['manufacturer'].value_counts()


manufacturer
other        220795
ford          62466
chevrolet     48600
toyota        30140
missing       14176
Name: count, dtype: int64

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [116]:
# Identify dataset into Features and target
features = ['vehicle_age','age_mileage','manufacturer','fuel','log_odometer','title_status','transmission','type','paint_color']
target = 'price'

# Subset dataframe
df_model = df[features + [target]].copy()

# Transform categorical variables into dummy 
categorical_cols = df_model.select_dtypes(include="object").columns.tolist()
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Split into X and y
X = df_model.drop(columns=[target])
y = df_model[target]

from sklearn.model_selection import train_test_split
# Split into Train-test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler

# Identify numeric columns (these are the ones to scale)
numeric_cols = ['vehicle_age', 'age_mileage', 'log_odometer']

# Create scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Transform test data using same scaler
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [119]:
#import regression libraries
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [120]:
#model 1: linear regression
lr_model= LinearRegression()
lr_model.fit(X_train,y_train)

LinearRegression()

In [121]:
#model 2: ridge regression
ridge_model = Ridge()
ridge_model.fit(X_train, y_train)

Ridge()

In [122]:
#model 3: lasso regression
lasso_model = Lasso()  
lasso_model.fit(X_train, y_train)

Lasso()

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

In [123]:
#import RMSE and R2 sklearn libraries
from sklearn.metrics import mean_squared_error, r2_score

#create evaluation function using RMSE and R2, where RMSE captures root mean squared error, and R2 represents the amount of variation captured by the variables
def evaluate_model(model,X_test,y_test):
    y_pred=model.predict(X_test)
    rmse=np.sqrt(mean_squared_error(y_test,y_pred))
    r2=r2_score(y_test,y_pred)
    return rmse,r2

In [124]:
#evaluating model 1: linear regression
rmse_lr,r2_lr=evaluate_model(lr_model,X_test,y_test)

In [125]:
#evaluating model 2: ridge regression
rmse_ridge,r2_ridge=evaluate_model(ridge_model,X_test,y_test)

In [126]:
#evaluating model 3: lasso regression
rmse_lasso,r2_lasso=evaluate_model(lasso_model,X_test,y_test)

In [127]:
#comparing RMSE & R2
print (f'Linear Model RMSE: {rmse_lr}, R2: {r2_lr}.\nRidge Model RMSE: {rmse_ridge}, R2: {r2_ridge}.\nLasso Model RMSE: {rmse_lasso}, R2: {r2_lasso}')

Linear Model RMSE: 0.8285365948390054, R2: 0.30784631145804386.
Ridge Model RMSE: 0.8285398199926834, R2: 0.3078409229058111.
Lasso Model RMSE: 0.9958914518697607, R2: -7.268661575521307e-06


### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

In [130]:
# Get feature names
feature_names = X_train.columns

# Get coefficients
coefficients = lr_model.coef_

# Create a DataFrame
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Abs_Coefficient": np.abs(coefficients)
})

# Sort by absolute value descending
coef_df = coef_df.sort_values(by="Abs_Coefficient", ascending=False).reset_index(drop=True)

# Show top 20 features
coef_df

,Feature,Coefficient,Abs_Coefficient
0,title_status_parts only,-1.543194,1.543194
1,fuel_hybrid,-0.741070,0.741070
2,title_status_missing,-0.717569,0.717569
3,fuel_gas,-0.680183,0.680183
4,fuel_electric,-0.541214,0.541214
5,fuel_other,-0.529522,0.529522
6,title_status_salvage,-0.514535,0.514535
7,transmission_other,0.470731,0.470731
8,type_truck,0.336466,0.336466
9,age_mileage,-0.325763,0.325763


In [ ]:
#Dummy variables use to split categorical features benchmark their coeffecients against a standard (i.e. the color coefficients of +/-1 would be against the benchmarked color Black)
